<a href="https://colab.research.google.com/github/iishikawalia/Machine-Learning/blob/main/Chapter2LLMfromScratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import urllib.request
url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x7e22fbd06990>)

In [4]:
with open("the-verdict.txt","r",encoding="utf-8") as f:
  raw_text=f.read()
print("Total number of characters: ",len(raw_text))
print(raw_text[0:99])

Total number of characters:  20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [5]:
import re
string="I am Ishika..."
new=re.split(r'(\s)',string)
print(new)

['I', ' ', 'am', ' ', 'Ishika...']


In [6]:
import re
string="I am Ishika..."
new=re.split(r'([,.]|\s)',string)
print(new)

['I', ' ', 'am', ' ', 'Ishika', '.', '', '.', '', '.', '']


In [7]:
result=[item for item in new if item.strip()]
print(result)

['I', 'am', 'Ishika', '.', '.', '.']


In [8]:
new_str="Hello, I'm Ishika - A COE student... Let's go --- yesss."
text=re.split(r'([,.:;?_!"()\']|--|\s)',new_str)
text=[item.strip() for item in result if item.strip()]
print(text)

['I', 'am', 'Ishika', '.', '.', '.']


In [9]:
preprocessed_text=re.split(r'([,.:;?_!"()\']|--|\s)',raw_text)
preprocessed_text=[item.strip() for item in preprocessed_text if item.strip()]
print(len(preprocessed_text))

4690


In [10]:
print(preprocessed_text[0:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


sorting and removing duplicates

In [11]:
all_words=sorted(set(preprocessed_text))
vocab_size=len(all_words)
print(vocab_size)

1130


converting to numeric representation, called token ID

In [12]:
vocab={token:integer for integer,token in enumerate(all_words)}
for i,item in enumerate(vocab.items()):
  print(item)
  if(i>=50):
    break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


In [13]:
class SimpleTokenizerV1:
  def __init__(self, vocab):
    self.str_to_int=vocab
    self.int_to_str={i:s for s,i in vocab.items()}
  def encode(self,text):
    preprocessed=re.split(r'([,.?!_"()\']|--|\s)',text)
    preprocessed=[item.strip() for item in preprocessed if item.strip()]
    ids=[self.str_to_int[s] for s in preprocessed]
    return ids
  def decode(self,ids):
    text=" ".join([self.int_to_str[i] for i in ids])
    text=re.sub(r'\s+([,.?!"()\'])',r'\1',text)
    return text

implementing encoding on an example text

In [14]:
tokenizer=SimpleTokenizerV1(vocab)
text="""It's the last he painted, you know,"
       Mrs. Gisburn said with pardonable pride."""
ids=tokenizer.encode(text)
print(ids)

[56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


implementing decoding on an example set of tokens

In [15]:
print(tokenizer.decode(ids))

It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


another example

In [16]:
text="Hello, I'm Ishika"
tt=tokenizer.encode(text)
print(tt)

KeyError: 'Hello'

In [17]:
all_tokens=sorted(list(set(preprocessed_text)))
all_tokens.extend(["<|endoftext|>","<|unk|>"])
vocab={token:integer for integer, token in enumerate(all_tokens)}

In [18]:
for i, item in enumerate(list(vocab.items())[-5:]):
  print(item)


('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [19]:
class SimpleTokenizerV2:
  def __init__(self, vocab):
    self.str_to_int=vocab
    self.int_to_str={i:s for s,i in vocab.items()}
  def encode(self,text):
    preprocessed_text=re.split(r'([,.:;?_!"()\']|--|\s)',text)
    preprocessed_text=[item.strip() for item in preprocessed_text if item.strip()]
    preprocessed_text=[item if item in self.str_to_int
         else "<|unk|>" for item in preprocessed_text]
    ids=[self.str_to_int[s] for s in preprocessed_text]
    return ids
  def decode(self,ids):
    text=" ".join([self.int_to_str[i] for i in ids])
    text=re.sub(r'\s+([,.:;?!"()\'])',r'\1',text)
    return text

In [20]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [21]:
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [22]:
print(tokenizer.decode(tokenizer.encode(text)))


<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.
